# Blue Book Encounter Phenomenology: OCR & Preprocessing Pipeline

Colab Pro optimized. Run on GPU runtime (A100 preferred, T4 workable).

**Pipeline:**
1. Download Blue Book PDFs from Internet Archive (decade ZIPs)
2. Copy PDFs to local SSD for fast I/O
3. Run Marker OCR on all documents (force_ocr, with checkpointing)
4. Extract and structure witness narratives
5. Build case index with metadata + clean text
6. Embed, cluster, and visualize
7. Save to Google Drive for persistence across sessions

## Cell 1: Environment Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import json

# Project directory on Drive (persists across sessions)
PROJECT_DIR = "/content/drive/MyDrive/blue_book_phenomenology"
PDF_DIR = f"{PROJECT_DIR}/pdfs"
CASE_DIR = f"{PROJECT_DIR}/cases"              # Extracted individual case PDFs
RAW_OCR_DIR = f"{PROJECT_DIR}/ocr_raw"         # Text from good-OCR PDFs
MARKER_OUT_DIR = f"{PROJECT_DIR}/marker_output"  # Marker re-OCR'd text
FINAL_CORPUS_DIR = f"{PROJECT_DIR}/corpus"       # Clean structured text
METADATA_DIR = f"{PROJECT_DIR}/metadata"

for d in [PROJECT_DIR, PDF_DIR, CASE_DIR, RAW_OCR_DIR, MARKER_OUT_DIR,
          FINAL_CORPUS_DIR, METADATA_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directory structure created on Google Drive.")

## Cell 2: Install Dependencies

In [ ]:
%%bash
# Marker (latest from datalab-to)
pip install marker-pdf -q

# NLP pipeline dependencies
pip install sentence-transformers umap-learn hdbscan bertopic -q
pip install spacy -q
python -m spacy download en_core_web_sm -q

# Utilities
pip install polars pyarrow tqdm requests beautifulsoup4 -q

echo "Dependencies installed."

## Cell 3: GPU Detection & Batch Size Configuration

In [ ]:
import torch

def get_gpu_config():
    """Detect GPU and set optimal Marker batch parameters."""
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Switch to GPU runtime.")

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9

    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

    # Marker uses ~5GB peak, ~3.5GB average per worker
    # Leave 2GB headroom for system
    max_workers = max(1, int((vram_gb - 2) / 5))

    # batch_multiplier scales internal batch sizes
    # Higher = faster but more VRAM
    if vram_gb >= 35:  # A100
        batch_multiplier = 4
        max_workers = min(max_workers, 6)  # Cap to avoid OOM
    elif vram_gb >= 14:  # T4 / V100
        batch_multiplier = 2
        max_workers = min(max_workers, 2)
    else:
        batch_multiplier = 1
        max_workers = 1

    config = {
        "gpu_name": gpu_name,
        "vram_gb": round(vram_gb, 1),
        "marker_workers": max_workers,
        "batch_multiplier": batch_multiplier,
        "estimated_pages_per_minute": max_workers * 6,
    }

    print(f"\nOptimal config:")
    print(f"  Marker workers: {max_workers}")
    print(f"  Batch multiplier: {batch_multiplier}")
    print(f"  Est. throughput: ~{config['estimated_pages_per_minute']} pages/min")

    return config

gpu_config = get_gpu_config()

## Cell 4: Download Blue Book Case Index from Black Vault

Black Vault organizes Blue Book by microfilm "rolls." Each roll is a large PDF containing multiple cases.

**Strategy:**
- First, download the case index to identify which rolls contain the 701 Unidentified cases
- Then download only those rolls
- This avoids downloading the entire 130K-page archive

**Important:** The 701 Unidentified cases list is widely available. Brad Sparks maintains the most authoritative version: "The Comprehensive Catalog of Project Blue Book Unknowns"

Download it manually and place at:
`/content/drive/MyDrive/blue_book_phenomenology/metadata/bb_unknowns.csv`

See: https://www.nicap.org/bb/BB_Unknowns.pdf

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm

unknowns_path = f"{METADATA_DIR}/bb_unknowns.csv"

if os.path.exists(unknowns_path):
    import polars as pl
    unknowns = pl.read_csv(unknowns_path)
    print(f"Loaded {len(unknowns)} Unidentified cases from index.")
else:
    print("WARNING: Unidentified cases index not found.")
    print(f"Place Brad Sparks' catalog at: {unknowns_path}")
    print("See: https://www.nicap.org/bb/BB_Unknowns.pdf")
    print("Or compile from NICAP's Blue Book archive.")

## Cell 5: Download Case Files from Internet Archive

The complete Project Blue Book archive (10,000+ case files) is hosted on the Internet Archive as decade-based ZIPs:

| File | Size | Coverage |
|------|------|----------|
| `1940s.zip` | 1.0 GB | 505 cases (1947-1949) |
| `1950s.zip` | 7.2 GB | ~4,000 cases |
| `1960s.zip` | 12.3 GB | ~5,500 cases |
| `19XXs.zip` | 651 MB | Undated/miscellaneous |

**Total:** ~21 GB, pre-segmented into individual case files.

Source: https://archive.org/details/bluebook

In [ ]:
import requests
import zipfile
import io

ARCHIVE_BASE = "https://archive.org/download/bluebook"
DECADE_ZIPS = ["1940s.zip", "1950s.zip", "1960s.zip", "19XXs.zip"]

CASE_DIR = f"{PROJECT_DIR}/cases"
os.makedirs(CASE_DIR, exist_ok=True)

def download_and_extract_decade(zip_name, output_dir):
    """Download a decade ZIP from Internet Archive and extract case PDFs."""
    decade = zip_name.replace(".zip", "")
    decade_dir = os.path.join(output_dir, decade)
    
    # Skip if already extracted
    if os.path.exists(decade_dir) and len(os.listdir(decade_dir)) > 0:
        count = len([f for f in os.listdir(decade_dir) if f.endswith('.pdf') or f.endswith('.PDF')])
        print(f"  {decade}: already extracted ({count} case files)")
        return decade_dir
    
    os.makedirs(decade_dir, exist_ok=True)
    url = f"{ARCHIVE_BASE}/{zip_name}"
    
    print(f"  Downloading {zip_name}...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    total_size = int(resp.headers.get('content-length', 0))
    
    # Download to temp file (ZIPs are too large to hold in memory)
    temp_path = os.path.join(output_dir, f"_temp_{zip_name}")
    with open(temp_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True,
                  desc=decade, leave=False) as pbar:
            for chunk in resp.iter_content(chunk_size=131072):
                f.write(chunk)
                pbar.update(len(chunk))
    
    # Extract
    print(f"  Extracting {zip_name}...")
    with zipfile.ZipFile(temp_path, 'r') as zf:
        zf.extractall(decade_dir)
    
    # Clean up temp file
    os.remove(temp_path)
    
    count = sum(1 for root, dirs, files in os.walk(decade_dir) 
                for f in files if f.lower().endswith('.pdf'))
    print(f"  {decade}: extracted {count} case files")
    return decade_dir

# Download all decades
print("Downloading complete Blue Book archive from Internet Archive...")
print("(~21 GB total — this will take a while on first run)\n")

case_dirs = []
for zip_name in DECADE_ZIPS:
    try:
        result = download_and_extract_decade(zip_name, CASE_DIR)
        case_dirs.append(result)
    except Exception as e:
        print(f"  ERROR downloading {zip_name}: {e}")

# Count total case files
all_pdfs = []
for d in case_dirs:
    for root, dirs, files in os.walk(d):
        for f in files:
            if f.lower().endswith('.pdf'):
                all_pdfs.append(os.path.join(root, f))

print(f"\nTotal case files: {len(all_pdfs)}")
downloaded = all_pdfs  # Rename for downstream cells

## Cell 6: Marker OCR (Local SSD + Checkpointing)

Copies PDFs to Colab's local SSD to avoid Drive I/O bottleneck, then runs Marker with `force_ocr=True` on all files. Each processed file is saved to Drive immediately as a checkpoint. On session reconnect, already-processed files are skipped automatically.

In [ ]:
import shutil
import subprocess
import glob

# --- Step 1: Copy PDFs from Drive to local SSD (bulk copy) ---
LOCAL_PDF_DIR = "/content/cases_local"
LOCAL_MARKER_OUT = "/content/marker_output_local"
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)
os.makedirs(LOCAL_MARKER_OUT, exist_ok=True)

# Use shell cp -r for bulk copy (much faster than Python per-file)
print("Copying PDFs from Drive to local SSD (bulk)...")
for decade in ["1940s", "1950s", "1960s", "19XXs"]:
    src = os.path.join(CASE_DIR, decade)
    if os.path.exists(src):
        print(f"  Copying {decade}...")
        subprocess.run(["cp", "-rn", src, LOCAL_PDF_DIR], check=True)
print("Copy complete.")

# Gather all local PDFs
local_pdfs = glob.glob(f"{LOCAL_PDF_DIR}/**/*.pdf", recursive=True) + \
             glob.glob(f"{LOCAL_PDF_DIR}/**/*.PDF", recursive=True)
print(f"Total PDFs on local SSD: {len(local_pdfs)}")

# --- Step 2: Find unprocessed files ---
to_process = []
skipped = 0
for pdf_path in local_pdfs:
    basename = os.path.splitext(os.path.basename(pdf_path))[0]
    drive_md = os.path.join(MARKER_OUT_DIR, f"{basename}.md")
    if os.path.exists(drive_md) and os.path.getsize(drive_md) > 0:
        skipped += 1
    else:
        to_process.append(pdf_path)

print(f"Already processed: {skipped}")
print(f"Remaining: {len(to_process)}")

if not to_process:
    print("All files already processed!")
else:
    # --- Step 3: Process in batches of 100 using Marker CLI with parallel workers ---
    BATCH_SIZE = 100
    N_WORKERS = min(4, gpu_config.get("marker_workers", 2))
    batches = [to_process[i:i+BATCH_SIZE] for i in range(0, len(to_process), BATCH_SIZE)]

    print(f"\nProcessing {len(to_process)} files in {len(batches)} batches of {BATCH_SIZE}")
    print(f"Using {N_WORKERS} parallel workers\n")

    for batch_idx, batch in enumerate(batches):
        batch_input = f"/content/_batch_input"
        batch_output = f"/content/_batch_output"
        # Clean and recreate
        if os.path.exists(batch_input):
            shutil.rmtree(batch_input)
        if os.path.exists(batch_output):
            shutil.rmtree(batch_output)
        os.makedirs(batch_input)
        os.makedirs(batch_output)

        for pdf_path in batch:
            link_path = os.path.join(batch_input, os.path.basename(pdf_path))
            if not os.path.exists(link_path):
                os.symlink(pdf_path, link_path)

        print(f"Batch {batch_idx + 1}/{len(batches)}: {len(batch)} files...")
        result = subprocess.run(
            [
                "marker", batch_input,
                "--output_dir", batch_output,
                "--workers", str(N_WORKERS),
                "--force_ocr",
                "--languages", "English",
            ],
            capture_output=True, text=True
        )

        if result.returncode != 0:
            print(f"  Marker stderr: {result.stderr[:500]}")

        # Copy output to Drive (checkpoint) and local
        copied = 0
        for root, dirs, files in os.walk(batch_output):
            for f in files:
                if f.endswith('.md'):
                    src = os.path.join(root, f)
                    shutil.copy2(src, os.path.join(MARKER_OUT_DIR, f))
                    shutil.copy2(src, os.path.join(LOCAL_MARKER_OUT, f))
                    copied += 1

        shutil.rmtree(batch_input)
        shutil.rmtree(batch_output)

        total_done = skipped + (batch_idx + 1) * BATCH_SIZE
        print(f"  Batch {batch_idx + 1} done: {copied} files -> Drive "
              f"({min(total_done, len(local_pdfs))}/{len(local_pdfs)} total)\n")

    print("All batches complete. Output saved to Drive.")

## Cell 8: Build Case Index

Since the archive is pre-segmented into individual case PDFs, this cell builds a unified index from all OCR'd text, tagging each case with its decade and filename-derived ID.

In [ ]:
import re
import polars as pl

print("Building case index from OCR'd text...")
all_cases = []

for md_dir in [MARKER_OUT_DIR, RAW_OCR_DIR]:
    if not os.path.exists(md_dir):
        continue
    for md_file in sorted(os.listdir(md_dir)):
        if not md_file.endswith('.md'):
            continue

        md_path = os.path.join(md_dir, md_file)
        with open(md_path, 'r', encoding='utf-8') as f:
            text = f.read()

        basename = md_file.replace('.md', '')
        
        # Try to extract case number from filename or text
        case_num_match = re.search(r'(\d+)', basename)
        case_id = case_num_match.group(1) if case_num_match else basename
        
        # Detect decade from parent directory or filename
        decade = "unknown"
        for d in ["1940s", "1950s", "1960s", "19XXs"]:
            if d in md_path:
                decade = d
                break

        all_cases.append({
            "case_id": case_id,
            "filename": basename,
            "decade": decade,
            "text": text,
            "char_count": len(text),
            "source": "marker" if MARKER_OUT_DIR in md_path else "pymupdf",
        })

print(f"Indexed {len(all_cases)} individual case documents.")

case_df = pl.DataFrame(all_cases)
case_df.write_parquet(f"{FINAL_CORPUS_DIR}/blue_book_cases.parquet")
print(f"Saved to {FINAL_CORPUS_DIR}/blue_book_cases.parquet")

# Summary by decade
for decade in sorted(case_df["decade"].unique().to_list()):
    subset = case_df.filter(pl.col("decade") == decade)
    print(f"  {decade}: {len(subset)} cases, median {subset['char_count'].median():.0f} chars")

## Cell 9: Witness Narrative Extraction

Within each case document, extract specifically the witness narrative sections. These are typically in:
- Section 11 of AF Form 112: "Brief summary of sighting"
- Attached witness statements
- Interview transcripts

This is a heuristic extraction -- Marker's layout detection helps but the forms are inconsistent across decades.

In [ ]:
def extract_witness_narrative(case_text):
    """
    Extract the witness narrative portion from a Blue Book case document.
    Returns the narrative text, or the full text if no clear section found.
    """
    narrative_patterns = [
        r'(?:BRIEF\s+SUMMARY\s+OF\s+SIGHTING)(.*?)(?:CONCLUSION|ANALYSIS|COMMENTS)',
        r'(?:DESCRIPTION\s+OF\s+SIGHTING)(.*?)(?:ACTION\s+TAKEN|CONCLUSION)',
        r'(?:NARRATIVE)(.*?)(?:CONCLUSION|ANALYSIS)',
        r'(?:WITNESS\s+STATEMENT)(.*?)(?:SIGNED|INVESTIGATOR)',
        r'(?:STATEMENT\s+OF\s+WITNESS)(.*?)(?:SIGNED|INVESTIGATOR)',
    ]

    for pattern in narrative_patterns:
        match = re.search(pattern, case_text, re.IGNORECASE | re.DOTALL)
        if match:
            narrative = match.group(1).strip()
            if len(narrative) > 100:
                return narrative

    # Fallback: return the longest paragraph-like section
    paragraphs = re.split(r'\n\s*\n', case_text)
    if paragraphs:
        longest = max(paragraphs, key=len)
        if len(longest) > 100:
            return longest.strip()

    return case_text

# Extract narratives
print("Extracting witness narratives...")
case_df = pl.read_parquet(f"{FINAL_CORPUS_DIR}/blue_book_cases.parquet")

narratives = []
for row in case_df.iter_rows(named=True):
    narrative = extract_witness_narrative(row["text"])
    narratives.append(narrative)

case_df = case_df.with_columns(
    pl.Series("witness_narrative", narratives)
)

case_df = case_df.with_columns(
    pl.col("witness_narrative").str.len_chars().alias("narrative_length")
)

# Filter to cases with meaningful narratives
viable = case_df.filter(pl.col("narrative_length") > 200)
print(f"\nViable cases with narratives > 200 chars: {len(viable)}")
print(f"Median narrative length: {viable['narrative_length'].median()} chars")

viable.write_parquet(f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet")
print(f"Saved to {FINAL_CORPUS_DIR}/blue_book_narratives.parquet")

## Cell 10: Quick Embedding Clustering (Layer 3 -- No Annotation Needed)

Instant-results cell. No annotation required. Encode all narratives with sentence-transformers, reduce with UMAP, cluster with HDBSCAN. See if Unidentified cases cluster differently.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading sentence-transformer model...")
model = SentenceTransformer('all-mpnet-base-v2')

# Load narratives
narratives_df = pl.read_parquet(f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet")
texts = narratives_df["witness_narrative"].to_list()

print(f"Encoding {len(texts)} narratives...")
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Save embeddings (expensive to recompute)
np.save(f"{FINAL_CORPUS_DIR}/narrative_embeddings.npy", embeddings)
print("Embeddings saved.")

# UMAP reduction
import umap

print("Running UMAP dimensionality reduction...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
coords_2d = reducer.fit_transform(embeddings)

# HDBSCAN clustering
import hdbscan

print("Running HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom'
)
cluster_labels = clusterer.fit_predict(coords_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_pct = (cluster_labels == -1).sum() / len(cluster_labels) * 100
print(f"\nFound {n_clusters} clusters ({noise_pct:.1f}% noise)")

## Cell 11: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels,
    cmap='Spectral',
    alpha=0.6,
    s=15,
    edgecolors='none'
)

ax.set_title("Blue Book Witness Narratives \u2014 Embedding Space", fontsize=14)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/embedding_clusters.png", dpi=150)
plt.show()

print(f"Plot saved to {PROJECT_DIR}/embedding_clusters.png")

## Cell 12: BERTopic for Cluster Interpretation

In [ ]:
from bertopic import BERTopic

print("Running BERTopic for cluster interpretation...")
topic_model = BERTopic(
    embedding_model=model,
    umap_model=reducer,
    hdbscan_model=clusterer,
    verbose=True
)

topics, probs = topic_model.fit_transform(texts, embeddings)

# Print topic summaries
print("\n" + "="*60)
print("TOPIC SUMMARIES")
print("="*60)
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    topic_info = topic_model.get_topic(topic_id)
    keywords = ", ".join([word for word, _ in topic_info[:8]])
    count = topics.count(topic_id)
    print(f"\nTopic {topic_id} ({count} cases): {keywords}")

# Save topic model
topic_model.save(f"{PROJECT_DIR}/bertopic_model")
print(f"\nTopic model saved to {PROJECT_DIR}/bertopic_model")

## Cell 13: Session Save & Resume Support

Save all state so you can resume in a new Colab session. Everything is on Google Drive already, but this creates a single checkpoint file for quick state recovery.

In [ ]:
checkpoint = {
    "gpu_config": gpu_config,
    "total_cases_extracted": len(all_cases) if 'all_cases' in dir() else 0,
    "viable_narratives": len(viable) if 'viable' in dir() else 0,
    "n_clusters": n_clusters if 'n_clusters' in dir() else 0,
    "embeddings_path": f"{FINAL_CORPUS_DIR}/narrative_embeddings.npy",
    "narratives_path": f"{FINAL_CORPUS_DIR}/blue_book_narratives.parquet",
    "topic_model_path": f"{PROJECT_DIR}/bertopic_model",
    "status": "complete_through_clustering",
}

with open(f"{PROJECT_DIR}/checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

print("Session checkpoint saved.")
print(f"\nTo resume in a new session:")
print(f"  1. Mount Google Drive")
print(f"  2. Load checkpoint from {PROJECT_DIR}/checkpoint.json")
print(f"  3. Load embeddings from {FINAL_CORPUS_DIR}/narrative_embeddings.npy")
print(f"  4. Load narratives from {FINAL_CORPUS_DIR}/blue_book_narratives.parquet")

## Next Steps (run in subsequent sessions)

1. **ANNOTATION (Layer 1):**
   - Export 100-150 narratives to Label Studio format
   - Annotate phenomenological spans (INITIAL_AWARENESS, etc.)
   - Fine-tune DeBERTa-v3-base for span classification

2. **AFFECT TRAJECTORIES (Layer 2):**
   - `pip install nrclex` or use GoEmotions model
   - Extract sentence-level VAD scores across each narrative
   - Compare trajectories between clusters

3. **MATCHED CONTROL ANALYSIS:**
   - Tag cases as Unidentified (701) vs Identified
   - Overlay USAF classification on embedding plot
   - Statistical tests: are Unidentified cases phenomenologically distinct from Identified cases in embedding space?

4. **CROSS-CORPUS TRANSFER:**
   - Apply the same pipeline to MUFON data, Colares reports, etc.
   - Test whether Blue Book phenomenological signatures generalize